# Artwork Style Explorer

**Question:** if a model looks at ~10k paintings *without being told their style*, does it rediscover the art movements?

Rule for this notebook: **never use `style` to fit anything.** Labels only come out at the end to grade the clusters.

Run order first: `python prepare_data.py` → `python embed.py` → this notebook → `streamlit run app.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

DATA = Path("../data")
df = pd.read_csv(DATA / "metadata.csv")
X = np.load(DATA / "embeddings.npy")   # (N, 512), each row = one painting
assert len(df) == len(X)
print(X.shape)
df["style"].value_counts()

## 0. Look at the data
TODO: show a grid of ~16 random thumbnails with their style as the title.
*Hint:* `plt.subplots(4, 4)`, `Image.open(DATA / "images" / file)`, `ax.axis("off")`.

In [ ]:
# TODO

## 1. PCA: how many dimensions actually matter? (course ch. 3)
TODO:
1. Fit `PCA()` on `X` and plot `explained_variance_ratio_.cumsum()`.
2. Pick the number of components that keeps ~90% of the variance (often 50–100).
3. Make `X_pca = PCA(n_components=...).fit_transform(X)`.

*Why:* clustering is faster and less noisy on fewer dimensions.

In [ ]:
from sklearn.decomposition import PCA

# TODO

## 2. k-means: find the groups (course ch. 1)
TODO:
1. Elbow plot: fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for k in `range(5, 41, 5)` and plot `inertia_`.
2. Pick k (try 27 = the number of real styles, and compare with your elbow choice).
3. Save the labels as `clusters`.

In [ ]:
from sklearn.cluster import KMeans

# TODO

## 3. Grade the clusters against real styles (course ch. 1, the fish crosstab)
TODO:
1. `pd.crosstab(clusters, df["style"])`, then show it as a heatmap (`plt.imshow` or `seaborn.heatmap`).
2. `adjusted_rand_score(df["style"], clusters)`: 0 = random, 1 = perfect.
3. Write 2–3 sentences: which styles are found cleanly (Ukiyo-e? Cubism?) and which get mixed up (Impressionism vs Post-Impressionism?). **This is the story of your project.**

In [ ]:
from sklearn.metrics import adjusted_rand_score

# TODO

## 4. Show what each cluster looks like
TODO: for 4–5 clusters, show 8 thumbnails each. Give each cluster a human name ("dark religious portraits", "Japanese prints"...).

In [ ]:
# TODO

## 5. Hierarchical clustering: which movements are related? (course ch. 2)
TODO:
1. Average the embeddings per style: `means = pd.DataFrame(X).groupby(df["style"]).mean()`. (Using labels here is fine: we're describing styles, not fitting a classifier.)
2. `linkage(means, method="average", metric="cosine")`, then `dendrogram(..., labels=means.index, orientation="left")`.
3. Does it rebuild art history? (Renaissance styles together? the Cubisms together?)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# TODO

## 6. t-SNE map (course ch. 2)
TODO:
1. `xy = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca)` (takes a few minutes).
2. Two scatter plots side by side: colored by `clusters`, and colored by style.

In [ ]:
from sklearn.manifold import TSNE

# TODO

## 7. Bonus: why embeddings? Try raw pixels
TODO: load the thumbnails as 32×32 grayscale, flatten (1024 numbers each), and run the same PCA → KMeans → ARI.
Compare the ARI with section 3. Expect it to be much worse. That's the argument for using CLIP.

In [ ]:
# TODO

## 8. Save results for the app
Run this once `xy` and `clusters` exist.

In [ ]:
np.save(DATA / "tsne.npy", xy)
np.save(DATA / "clusters.npy", np.asarray(clusters))
print("Saved. Now run: streamlit run app.py")